# Camada 2 — regras de sinalização e janelas de proximidade

Testa os indicadores da mediação setorizada sob:

- **R1**: 1 termo nacional e 1 termo setorizado no discurso. Reproduz 86,8% / 76,5% / 80,8%.
- **R2**: em cada lado, pelo menos uma categoria com 2 termos distintos ou 3 ocorrências.
- **Janela de 20 tokens**: termo nacional e setorizado a no máximo 20 tokens de distância.
- **Janela de 50 tokens**: idem, com 50 tokens.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` a partir dos dados da API da Câmara dos Deputados (ver `data/README_DATA.md`). O notebook procura essa pasta no Google Drive montado.

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*.


In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_camada2_regras'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # percentil do limiar fixo da R3 (0.50 = mediana)

In [ ]:
# 3) Ler os discursos pré-processados (mesma leitura do script 14)
import pandas as pd
from collections import Counter
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / 'corpus_artigo1_metadata.csv', usecols=['id_legislatura'])
textos, legs = [], []
for leg in LEGS:
    linhas = (PRE / 'by_legislature' / f'leg{leg}.txt').read_text(encoding='utf-8').splitlines()
    n_meta = int((meta['id_legislatura'] == leg).sum())
    if n_meta != len(linhas):
        raise ValueError(f'Leg {leg}: metadados={n_meta} texto={len(linhas)}')
    textos += linhas; legs += [leg] * len(linhas)
    print(f'Legislatura {leg}: {len(linhas):,} discursos')
df = pd.DataFrame({'_leg': legs, '_tokens': [t.split() for t in textos]})
del textos, meta
print(f'TOTAL: {len(df):,} discursos  (dissertação: 439.627)')

In [ ]:
# 4) Dicionários
SETORIZADO = {
 "agricultura_rural": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","pecuaria","safra","assentamento"},
 "empresas_industria": {"empresa","empresas","industria","industrial","comercio","comerciante","empreendedor","empreendedores","microempresa","empresarios"},
 "trabalhadores_sindicatos": {"trabalhador","trabalhadores","sindicato","sindicatos","servidor","servidores","categoria","categorias","funcionario","funcionarios"},
 "servidores_profissoes": {"professor","professores","medico","medicos","enfermeiro","enfermeiros","policial","policiais","militar","militares"},
 "mulheres_familia_infancia": {"mulher","mulheres","familia","familias","crianca","criancas","adolescente","adolescentes","juventude","idosos","idoso"},
 "raca_indigenas_minorias": {"negros","negro","racismo","indigena","indigenas","quilombolas","quilombola","lgbt","deficiencia","pessoa","pessoas"},
 "religiao_moral": {"religiao","religioso","religiosos","igreja","igrejas","evangelicos","catolicos","familia","vida","aborto"},
 "territorio_regiao": {"municipio","municipios","prefeito","prefeitos","nordeste","norte","amazonia","semiarido","regional","regiao","comunidade","comunidades"},
 "forcas_seguranca": {"policia","policial","policiais","militar","militares","bombeiros","guarda","forcas","armadas"},
}
TERMOS_SET = set().union(*SETORIZADO.values())

NAC_A = {
 "educacao": {"educacao","escola","ensino","professor","professores","universidade","fundeb"},
 "saude": {"saude","sus","hospital","medico","medicos","vacina","pandemia","covid"},
 "economia": {"economia","emprego","trabalho","renda","salario","inflacao","juros","credito"},
 "seguranca": {"seguranca","violencia","crime","policia","armas","trafico","homicidio"},
 "infraestrutura": {"infraestrutura","transporte","rodovia","estrada","energia","saneamento","internet"},
 "meio_ambiente": {"ambiente","ambiental","amazonia","floresta","desmatamento","clima","agua"},
 "direitos_politica_social": {"direitos","cidadania","pobreza","fome","desigualdade","assistencia","previdencia"},
 "instituicoes": {"democracia","constituicao","congresso","camara","senado","governo","voto","judiciario"},
}
NAC_B = {k: v - TERMOS_SET for k, v in NAC_A.items()}

CAMADA1 = {
 "Educação": {"educacao","escola","escolas","ensino","professor","professores","aluno","alunos","universidade","universidades","creche","alfabetizacao","fundeb","enem","estudante","estudantes","pesquisa","ciencia","cientifico","tecnologia"},
 "Saúde": {"saude","sus","hospital","hospitais","medico","medicos","enfermeiro","enfermeiros","enfermeiras","vacina","vacinas","pandemia","covid","doenca","doencas","tratamento","medicamento","medicamentos","pacientes","atendimento"},
 "Economia, trabalho e renda": {"economia","economico","economica","emprego","empregos","trabalho","trabalhador","trabalhadores","renda","salario","salarios","inflacao","juros","credito","industria","empresas","empresa","mercado","crescimento","desenvolvimento","investimento","investimentos","tributaria","imposto","impostos"},
 "Segurança pública e violência": {"seguranca","violencia","crime","crimes","criminalidade","policia","policial","policiais","prisao","presidio","armas","arma","trafico","homicidio","homicidios","feminicidio","militar","militares"},
 "Meio ambiente e clima": {"meio","ambiente","ambiental","amazonia","floresta","florestas","desmatamento","clima","climatica","climatico","sustentavel","sustentabilidade","biodiversidade","indigena","indigenas","garimpo","queimadas","agua","saneamento"},
 "Infraestrutura, transporte e habitação": {"infraestrutura","transporte","transportes","rodovia","rodovias","estrada","estradas","ferrovia","ferrovias","porto","portos","aeroporto","aeroportos","energia","eletrica","habitacao","moradia","saneamento","internet","telecomunicacoes","obra","obras"},
 "Proteção social e desigualdade": {"pobreza","fome","desigualdade","desigualdades","assistencia","social","bolsa","familia","beneficio","beneficios","previdencia","aposentadoria","aposentados","vulnerabilidade","moradia","direitos","mulheres","criancas","juventude","idosos","deficiencia"},
 "Democracia e instituições": {"democracia","democratico","democratica","constituicao","constitucional","congresso","camara","senado","parlamento","governo","presidente","presidencia","republica","eleicao","eleicoes","voto","impeachment","stf","supremo","judiciario","justica"},
 "Direitos e cidadania": {"direito","direitos","cidadania","cidadao","cidadaos","igualdade","liberdade","justica","mulher","mulheres","negros","racismo","discriminacao","lgbt","indigena","indigenas","quilombolas","criancas","adolescentes"},
}
NAC_C = {k: v - TERMOS_SET for k, v in CAMADA1.items()}

TERMOS = {"A": set().union(*NAC_A.values()), "B": set().union(*NAC_B.values()), "C": set().union(*NAC_C.values())}
print("Termos retirados em B:", sorted(TERMOS["A"] - TERMOS["B"]))
print("Termos de agenda nacional -> A:", len(TERMOS["A"]), "| B:", len(TERMOS["B"]), "| C:", len(TERMOS["C"]))

In [ ]:
# 5) Sinalização por discurso
import numpy as np
from bisect import bisect_left

def r2(cont, dic):
    for termos in dic.values():
        hits = [t for t in termos if cont[t] > 0]
        if len(hits) >= 2 or sum(cont[t] for t in hits) >= 3:
            return True
    return False

def perto(pos_a, pos_b, k):
    if not pos_a or not pos_b:
        return False
    for p in pos_a:
        i = bisect_left(pos_b, p)
        if i < len(pos_b) and pos_b[i] - p <= k:
            return True
        if i > 0 and p - pos_b[i - 1] <= k:
            return True
    return False

TN = TERMOS["A"]
n = len(df)
nac1 = np.zeros(n, bool); set1 = np.zeros(n, bool)
nac2 = np.zeros(n, bool); set2 = np.zeros(n, bool)
j20 = np.zeros(n, bool); j50 = np.zeros(n, bool)
for i, toks in enumerate(df["_tokens"]):
    pn = [k for k, t in enumerate(toks) if t in TN]
    ps = [k for k, t in enumerate(toks) if t in TERMOS_SET]
    nac1[i], set1[i] = bool(pn), bool(ps)
    if pn or ps:
        cont = Counter(toks)
        nac2[i] = r2(cont, NAC_A)
        set2[i] = r2(cont, SETORIZADO)
    if pn and ps:
        j50[i] = perto(pn, ps, 50)
        if j50[i]:
            j20[i] = perto(pn, ps, 20)
    if i % 50_000 == 0:
        print(f"  {i:,}/{n:,}")
print("Pronto.")

In [ ]:
# 6) Indicadores gerais
regras = {
    "R1":       (nac1, set1, nac1 & set1),
    "R2":               (nac2, set2, nac2 & set2),
    "Janela 50 tokens": (nac1, set1, j50),
    "Janela 20 tokens": (nac1, set1, j20),
}
linhas = []
for nome, (nac, sett, mis) in regras.items():
    linhas.append({"regra": nome,
                   "agenda_nacional_%": round(100 * nac.mean(), 1),
                   "setorizada_%": round(100 * sett.mean(), 1),
                   "mistos_%": round(100 * mis.mean(), 1),
                   "setorizada_entre_nacionais_%": round(100 * mis.sum() / nac.sum(), 1)})
geral = pd.DataFrame(linhas).set_index("regra")
print("Dissertação (R1): nacional 86,8 | setorizada 76,5 | mistos 70,1 | setorizada entre nacionais 80,8")
geral

In [ ]:
# 7) Setorizada entre nacionais, por legislatura
leg = df["_leg"].values
tab = {}
for nome, (nac, sett, mis) in regras.items():
    tab[nome] = [round(100 * mis[leg == L].sum() / nac[leg == L].sum(), 1) for L in LEGS]
por_leg = pd.DataFrame(tab, index=LEGS)
por_leg.index.name = "legislatura"
por_leg

In [ ]:
# 8) Salvar no Drive
geral.to_csv(SAIDA / "camada2_regras_geral.csv", encoding="utf-8-sig")
por_leg.to_csv(SAIDA / "camada2_regras_por_legislatura.csv", encoding="utf-8-sig")
print("Arquivos salvos em:", SAIDA)